In [2]:
%pip install sentence-transformers scikit-learn spacy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------- -------------------------------- 2.4/12.8 MB 15.4 MB/s eta 0:00:01
     ------------------- -------------------- 6.3/12.8 MB 17.5 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 18.6 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 18.6 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 13.7 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [5]:
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\mayank goyal\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
model = SentenceTransformer('BAAI/bge-m3')

c:\Users\mayank goyal\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mayank goyal\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 14453.81it/s]


In [7]:
def process_article(text):
    """Cleans and splits text into meaningful sentences."""
    doc = nlp(text)
    # Filter out very short 'noise' sentences (less than 30 chars)
    return [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 30]

In [8]:
def align_sentences(article_1_text, article_2_text, threshold=0.75):
    """Matches sentences between two articles based on semantic meaning."""
    
    sents1 = process_article(article_1_text)
    sents2 = process_article(article_2_text)
    
    # 2. Generate Embeddings
    embeddings1 = model.encode(sents1)
    embeddings2 = model.encode(sents2)
    
    # 3. Calculate Cosine Similarity Matrix
    # This creates a 'Heatmap' of how every sentence matches every other sentence
    sim_matrix = cosine_similarity(embeddings1, embeddings2)
    
    matches = []
    
    # 4. Simple Greedy Matching (Bipartite Matching logic)
    for i, row in enumerate(sim_matrix):
        best_match_idx = np.argmax(row)
        highest_score = row[best_match_idx]
        
        if highest_score >= threshold:
            matches.append({
                "source_sent": sents1[i],
                "target_sent": sents2[best_match_idx],
                "score": float(highest_score)
            })
            
    return matches

In [9]:
art1 = "The police dispersed the aggressive crowd using forceful measures late last night."
art2 = "Officers cleared the gathering of peaceful protestors using tear gas during the evening."

results = align_sentences(art1, art2)

for m in results:
    print(f"Match (Score: {m['score']:.2f}):")
    print(f"A: {m['source_sent']}")
    print(f"B: {m['target_sent']}\n")

Match (Score: 0.76):
A: The police dispersed the aggressive crowd using forceful measures late last night.
B: Officers cleared the gathering of peaceful protestors using tear gas during the evening.



In [10]:
def highlight_spin(sent1, sent2):
    doc1 = nlp(sent1)
    doc2 = nlp(sent2)
    
    # Extract adjectives (the primary source of spin)
    adj1 = [token.text.lower() for token in doc1 if token.pos_ == "ADJ"]
    adj2 = [token.text.lower() for token in doc2 if token.pos_ == "ADJ"]
    
    print(f"--- Spin Detection ---")
    print(f"Article A Framing: {adj1}") # ['aggressive', 'forceful']
    print(f"Article B Framing: {adj2}") # ['peaceful']
    
    # Logic: If an adjective in A is NOT in B, it's likely "Spin"
    spin_words = set(adj1) ^ set(adj2) 
    return list(spin_words)

# Try it out
diffs = highlight_spin(results[0]['source_sent'], results[0]['target_sent'])
print(f"🚩 Highlight these for the journalist: {diffs}")

--- Spin Detection ---
Article A Framing: ['aggressive', 'forceful', 'last']
Article B Framing: ['peaceful', 'tear']
🚩 Highlight these for the journalist: ['last', 'peaceful', 'tear', 'forceful', 'aggressive']


In [11]:
import requests
from bs4 import BeautifulSoup

In [12]:
def fetch_article_text(url):
    headers = {'User-Agent': 'Mozilla/5.0'} # Pretend to be a browser
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Pro Tip: Most news sites put content in <p> tags inside an <article> or <div>
    paragraphs = soup.find_all('p')
    full_text = " ".join([p.get_text() for p in paragraphs])
    
    return full_text

In [15]:
import requests
from bs4 import BeautifulSoup

def fetch_article_text(url):
    # 🕵️‍♂️ The 'Stealth' Header Pack
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Referer': 'https://www.google.com/', # Tell them Google sent you
        'DNT': '1', # Do Not Track
    }
    
    try:
        # Increase timeout to 15s in case the site is slow
        response = requests.get(url, headers=headers, timeout=15)
        
        # If it's still 500, the site might be blocking this specific IP
        if response.status_code == 500:
            return "🚩 Error 500: The site is blocking our bot. Try a different news source link!"
            
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # BBC and modern news sites often use <article> tags
        # We'll try to find the main content more precisely
        article = soup.find('article')
        if article:
            paragraphs = article.find_all('p')
        else:
            paragraphs = soup.find_all('p')
            
        full_text = " ".join([p.get_text().strip() for p in paragraphs if len(p.get_text()) > 30])
        return full_text
        
    except Exception as e:
        return f"Error: {str(e)}"

# --- TEST DRIVE AGAIN ---
test_url = "https://www.bbc.com/news/world-us-canada-68594191"
print(fetch_article_text(test_url)[:500])

Error: 404 Client Error: Not Found for url: https://www.bbc.com/news/world-us-canada-68594191


In [19]:
# 1. Provide a real news URL (Example: a BBC or CNN link)
test_url = "https://www.cnbc.com/2026/03/20/us-issues-30-day-sanctions-waiver-for-sale-of-iranian-oil-at-sea.html" 

# 2. Call your function and store the result
article_content = fetch_article_text(test_url)

# 3. Print the first 500 characters to see if it worked!
print("--- Scraped Content Preview ---")
print(article_content[:500] + "...")

--- Scraped Content Preview ---
The Trump administration on Friday issued a 30-day sanctions waiver for the purchase of Iranian oil at sea to ease energy supply pressures since the start of the U.S.-Israeli war on Iran, U.S. Treasury Secretary Scott Bessent said. This was the third time the U.S. has temporarily waived sanctions in about two weeks. The U.S. had previously eased sanctions on Russian oil and on Friday issued a general license allowing the sale of Iranian crude oil and petroleum products loaded on vessels as of Ma...


In [17]:
# Use a more stable, scraper-friendly link
test_url = "https://www.thehindu.com/news/international/us-issues-30-day-sanctions-waiver-for-sale-of-iranian-oil-at-sea/article70768207.ece"

article_content = fetch_article_text(test_url)

if "Error" in article_content:
    print(article_content)
else:
    print("✅ SUCCESS! We broke through the wall.")
    print("--- Content Preview (First 300 chars) ---")
    print(article_content[:300] + "...")

✅ SUCCESS! We broke through the wall.
--- Content Preview (First 300 chars) ---
The View From India
									Looking at World Affairs from the Indian perspective. First Day First Show
									News and reviews from the world of cinema and streaming. Today's Cache
									Your download of the top 5 technology stories of the day. Science For All
									The weekly newsletter fr...


In [18]:
def clean_news_text(raw_text):
    # 1. Split into lines
    lines = raw_text.split("...") # Or based on your separator
    
    # 2. Filter: Only keep 'Real' sentences
    # Rule: Must be > 60 chars (news sentences are long) and end with a period.
    cleaned = [s.strip() for s in raw_text.split('.') if len(s.strip()) > 60]
    
    # 3. Join back into a clean block
    return ". ".join(cleaned)

# Let's test the filter on your 'Success' data
clean_article = clean_news_text(article_content)
print("--- Cleaned Story (The Real Meat) ---")
print(clean_article[:500] + "...")

--- Cleaned Story (The Real Meat) ---
The View From India
									Looking at World Affairs from the Indian perspective. First Day First Show
									News and reviews from the world of cinema and streaming. Today's Cache
									Your download of the top 5 technology stories of the day. Science For All
									The weekly newsletter from science writers takes the jargon out of science and puts the fun in! Data Point
									Decoding the headlines with facts, figures, and numbers Health Matters
									Ramya Kannan writes to you o...


In [20]:
# Fetch the CNBC content
cnbc_url = "https://www.cnbc.com/2026/03/20/us-issues-30-day-sanctions-waiver-for-sale-of-iranian-oil-at-sea.html"
raw_cnbc_text = fetch_article_text(cnbc_url)

# Apply our 'Senior' cleaning logic
def get_clean_prose(text):
    # Split by periods, keep sentences that look like actual reporting
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 50]
    # Filter out common 'noise' phrases
    blacklist = ["subscription", "copyright", "all rights reserved", "follow us", "click here"]
    clean_sentences = [s for s in sentences if not any(word in s.lower() for word in blacklist)]
    return clean_sentences

cnbc_sentences = get_clean_prose(raw_cnbc_text)

print(f"✅ Successfully extracted {len(cnbc_sentences)} clean sentences from CNBC.")
print(f"Sample Fact: {cnbc_sentences[0]}...")

✅ Successfully extracted 9 clean sentences from CNBC.
Sample Fact: The Trump administration on Friday issued a 30-day sanctions waiver for the purchase of Iranian oil at sea to ease energy supply pressures since the start of the U...


In [21]:
# 1. Fetch and Clean both articles
url_a = "https://www.cnbc.com/2026/03/20/us-issues-30-day-sanctions-waiver-for-sale-of-iranian-oil-at-sea.html"
url_b = "https://www.freemalaysiatoday.com/category/business/2026/03/21/us-allows-30-day-sale-of-iran-oil-at-sea-in-bid-to-tame-prices"

text_a = fetch_article_text(url_a)
text_b = fetch_article_text(url_b)

sents_a = get_clean_prose(text_a)
sents_b = get_clean_prose(text_b)

# 2. Run the Alignment Engine (using our logic from earlier)
matches = align_sentences(". ".join(sents_a), ". ".join(sents_b))

# 3. Display the Results
print(f"🔍 Found {len(matches)} Fact-Matches between CNBC and Free Malaysia Today.\n")

for i, m in enumerate(matches[:3]): # Let's look at the top 3
    print(f"--- MATCH #{i+1} (Confidence: {m['score']:.2f}) ---")
    print(f"CNBC: {m['source_sent']}")
    print(f"FMT : {m['target_sent']}\n")

🔍 Found 3 Fact-Matches between CNBC and Free Malaysia Today.

--- MATCH #1 (Confidence: 0.75) ---
CNBC: The Trump administration on Friday issued a 30-day sanctions waiver for the purchase of Iranian oil at sea to ease energy supply pressures since the start of the U. has temporarily waived sanctions in about two weeks.
FMT : The US previously eased sanctions on Russian oil and on Friday issued a general license allowing the sale of Iranian crude oil and petroleum products loaded on vessels by Friday.

--- MATCH #2 (Confidence: 0.90) ---
CNBC: had previously eased sanctions on Russian oil and on Friday issued a general license allowing the sale of Iranian crude oil and petroleum products loaded on vessels as of March 20 to April 19, according to the license posted to the Treasury Department's website.
FMT : The US previously eased sanctions on Russian oil and on Friday issued a general license allowing the sale of Iranian crude oil and petroleum products loaded on vessels by Friday.

-

In [22]:
def get_unique_words(sent1, sent2):
    # Use spaCy to get clean tokens, ignoring punctuation/stopwords
    words1 = {t.text.lower() for t in nlp(sent1) if not t.is_stop and t.is_alpha}
    words2 = {t.text.lower() for t in nlp(sent2) if not t.is_stop and t.is_alpha}
    
    unique_to_cnbc = words1 - words2
    unique_to_fmt = words2 - words1
    
    return unique_to_cnbc, unique_to_fmt

# Test on Match #1
u_cnbc, u_fmt = get_unique_words(matches[0]['source_sent'], matches[0]['target_sent'])

print(f"🚩 CNBC's Unique Spin: {u_cnbc}")
print(f"🚩 FMT's Unique Spin: {u_fmt}")

🚩 CNBC's Unique Spin: {'temporarily', 'waived', 'trump', 'energy', 'pressures', 'start', 'supply', 'day', 'weeks', 'purchase', 'waiver', 'ease', 'administration', 'sea'}
🚩 FMT's Unique Spin: {'crude', 'license', 'previously', 'sale', 'vessels', 'petroleum', 'loaded', 'russian', 'general', 'allowing', 'eased', 'products'}
